# store

> The `learning` shelf: concepts, attempts and sessions as vault documents.

This module keeps all learning data in your vault, on one shelf named `learning`.
Each concept, attempt and session is one vault document. Thus each tool that reads the vault can search it.
The other modules read and write the shelf only through the `Learning` class in this module.

In [ ]:
#| default_exp store

In [ ]:
#| hide
from fastcore.test import *

In [ ]:
#| hide
import os, tempfile
os.environ['VISHALAKSHI_VAULT'] = os.environ['ABHYASA_VAULT'] = os.path.join(tempfile.mkdtemp(), 'vault.db')   # a test that opens the default vault gets a temporary one

Import the tools that this module uses, and the `Vault` class of the vault.

In [ ]:
#| export
import json, logging, re, threading, time, uuid, hashlib, unicodedata
from pathlib import Path
from fastcore.utils import *
from vishalakshi import Vault

log = logging.getLogger(__name__)

## The shelf

Search reads the text of a document. The `meta` JSON keeps the state.
`Vault.set_meta` changes the state and does not embed the text again.
The table gives the source URI of each kind of document.

| kind | source |
|---|---|
| `concept` | `learning://concept/<slug>` |
| `attempt` | `learning://attempt/<id>` |
| `session` | `learning://session/<id>` |

These constants name the shelf, give the start state of a concept, and list the fields that merge.

In [ ]:
#| export
SHELF = 'learning'
#: the learner state of a concept that has no attempts yet
STATE = dict(floor=0, ceiling=6, stability=0., due=None, last=None, streak=0, misses=0, seen=0, shaky=None, misconceptions=[])
#: content fields that a new source adds to, and does not replace
LISTS = ('prereqs', 'related', 'aliases', 'provenance', 'questions')
_SKIP = ('content_hash', 'pii_scanned')

`slugify` makes the key of a concept from its name. It changes a name in an Indic script, such as Devanagari, to IAST first. It changes `+`, `#` and `&` to words, so C, C++ and C# get different slugs. A name with no letters that it can read gets a short hash of the name. Thus the same name always gives the same slug. `_src` makes the source URI of a document.

In [ ]:
#| export
SYMBOLS = {'+': ' plus ', '#': ' sharp ', '&': ' and '}

def _latin(w:str) -> str:
    "`w` in IAST when it is in an Indic script such as Devanagari or Telugu; else `w`."
    if w.isascii(): return w
    from vidyut.lipi import transliterate, detect, Scheme
    d = detect(w)
    return transliterate(w, d, Scheme.Iast) if d and d not in (Scheme.Iast, Scheme.Iso15919) else w

def slugify(s:str) -> str:
    "Make a lower-case ASCII slug from `s`: `saṃdhi` and `संधि` give `samdhi`, and `C++` gives `c-plus-plus`."
    t = re.sub(r'[+#&]', lambda m: SYMBOLS[m[0]], ' '.join(map(_latin, s.split())))
    t = unicodedata.normalize('NFKD', t).encode('ascii', 'ignore').decode()
    return re.sub(r'[^a-z0-9]+', '-', t.lower()).strip('-') or 'c-' + hashlib.sha1(s.encode()).hexdigest()[:8]

def usable_name(name) -> bool:
    "Whether `name` has two characters at least, and a letter or a digit. A name such as `x` is not an idea."
    n = str(name or '').strip()
    return len(n) >= 2 and any(ch.isalnum() for ch in n)

def _src(kind, slug): return f'learning://{kind}/{slug}'

`_union` merges two lists and keeps the order. `concept_text` makes the text that search reads.

In [ ]:
#| export
def _union(a, b):
    "Items of `a`, then the new items of `b`, in order, compared as JSON."
    seen, out = set(), []
    for x in (*(a or ()), *(b or ())):
        k = json.dumps(x, sort_keys=True, default=str)
        if k not in seen: seen.add(k); out.append(x)
    return out

def concept_text(c) -> str:
    "The searchable text of a concept: its name, statement, aliases and subject."
    al = ', '.join(c.get('aliases') or [])
    return '\n\n'.join(o for o in (c['name'], c.get('statement', ''), al and f'Also: {al}',
                                   c.get('subject') and f'Subject: {c["subject"]}') if o)

`Learning` opens one shelf of a vault. It also accepts a path, or `None` for the default vault.

In [ ]:
#| export
class Learning:
    "Concepts, attempts and sessions on one shelf of your vault."
    def __init__(self, vault=None, shelf:str=SHELF):
        if not isinstance(vault, Vault): vault = Vault(vault)
        self.vault, self.v = vault, vault.shelf(shelf)
        self._lock = threading.RLock()   # a change of a group is one step: look, then write
        try: self.repair_names()   # a name that is a slug becomes words, one time
        except Exception as e: log.warning('abhyasa could not repair the names of the concepts: %r', e)
        try: import abhyasa.journey; self.repair_titles()   # a walk titled with an arXiv id gets the title of its paper, one time
        except Exception as e: log.warning('abhyasa could not repair the titles of the walks: %r', e)
        try: getattr(self, 'repair_subjects', lambda: 0)()   # the `journey` module adds it: a walk gives its subject to its concepts, one time
        except Exception as e: log.warning('abhyasa could not repair the subjects of the concepts: %r', e)

    def _docs(self, kind, where='', args=()):
        w = 'kind=?' + (f' and ({where})' if where else '')
        return self.v.t.docs(where=w, where_args=[kind, *args])

    def _row(self, r):
        m = {k: v for k, v in json.loads(r['meta'] or '{}').items() if k not in _SKIP}
        return AttrDict({**STATE, **m, 'doc_id': r['id'], 'name': m.get('name', r['title'])})

    def __repr__(self): return f'Learning({self.v.path!r}: {len(self._docs("concept"))} concepts)'

## Read concepts

Each method returns concepts as `AttrDict`s with the learner state filled in.

In [ ]:
#| export
@patch
def concept(self:Learning, slug:str) -> AttrDict|None:
    "One concept by slug, with its learner state."
    r = first(self.v.t.docs(where='source=?', where_args=[_src('concept', slug)]))
    return self._row(r) if r else None

@patch
def concept_by_doc(self:Learning, doc_id:str) -> AttrDict|None:
    "The concept that a search hit (its `doc_id`) belongs to, or None when the hit is not a concept."
    r = first(self.v.t.docs(where="id=? and kind='concept'", where_args=[doc_id]))
    return self._row(r) if r else None

@patch
def concepts(self:Learning, subject:str=None) -> L:
    "All concepts, or the concepts of one subject."
    rs = self._docs('concept', *(("json_extract(meta, '$.subject')=?", [subject]) if subject else ()))
    return L(rs).map(self._row)

@patch
def subjects(self:Learning) -> list:
    "The subjects that have concepts, with counts."
    return self.v.db.q(f"select json_extract(meta, '$.subject') as subject, count(*) as n from [{self.v.t.prefix}docs] "
                       "where kind='concept' group by 1 order by 2 desc")

## Write concepts

`put` adds a concept or adds to it. It keeps the learner state, merges the list fields and replaces the other fields.
It does not change the group of a concept that the learner moved by hand (see Groups below).

In [ ]:
#| export
@patch
def put(self:Learning,
        name:str,              # the concept's name
        statement:str='',      # a true, self-contained statement of the idea
        subject:str='',        # e.g. 'sanskrit', 'jyotisha', 'ml'
        slug:str=None,         # defaults to `slugify(name)`
        prereqs=(),            # slugs this concept needs first
        related=(),            # slugs linked to it, in any direction
        aliases=(),            # other names
        provenance=(),         # where it came from: dicts such as `{'src': 'vedicreader', 'ref': ...}`
        questions=(),          # ready questions: `{'q':..., 'a':..., 'depth':...}`
        **meta) -> AttrDict:
    "Add a concept or add to one. Keep the learner state, merge the list fields and replace the other fields. A name that you set by hand stays."
    with self._lock:   # one step: look at the old concept, then write
        slug = slug or slugify(name)
        old = self.concept(slug) or {}
        if old.get('named_by') == 'hand' and meta.get('named_by') != 'hand': name = old['name']   # a name that you set stays: use `rename`
        if old.get('group_by') in ('hand', 'sort', 'placed', 'undone'): meta.pop('group', None); meta.pop('group_by', None)   # a group that you, a saved sort or a placement set stays: use `set_group`
        elif 'group' in meta: meta['group'] = clean_group(meta['group'], subject or old.get('subject', ''))
        new = dict(name=name, statement=statement or old.get('statement', ''), subject=subject or old.get('subject', ''), slug=slug, **meta)
        for k, v in dict(prereqs=prereqs, related=related, aliases=aliases, provenance=provenance, questions=questions).items():
            new[k] = _union(old.get(k), v)
        new['prereqs'] = [p for p in new['prereqs'] if p != slug]
        m = {**STATE, **{k: v for k, v in old.items() if k != 'doc_id'}, **new}
        r = self.v.add(concept_text(m), title=name, source=_src('concept', slug), kind='concept', meta=m)
        if not r.get('changed'): self.v.set_meta(r['doc_id'], **m)
        return self.concept(slug)

`update` and `link` change one concept. `drop` removes it.

In [ ]:
#| export
@patch
def update(self:Learning, slug:str, **kv) -> AttrDict:
    "Change the state of a concept and do not embed its text again."
    c = self.concept(slug)
    if c is None: raise KeyError(f'no concept {slug!r}')
    self.v.set_meta(c.doc_id, **kv)
    return self.concept(slug)

@patch
def link(self:Learning, slug:str, prereq:str):
    "Record that `prereq` comes before `slug`."
    c = self.concept(slug)
    if c is None: raise KeyError(f'no concept {slug!r}')
    if prereq != slug and prereq not in c.prereqs: self.update(slug, prereqs=[*c.prereqs, prereq])

@patch
def drop(self:Learning, slug:str):
    "Remove a concept. Its attempts stay as a record of what happened."
    if (c := self.concept(slug)): self.v.forget(c.doc_id)

## Move old slugs

abhyasa 0.0.5 and older made slugs in a different way. A name in Devanagari got a random slug, and C, C++ and C# got the same slug, `c`.
`reslug` finds each concept that has an old slug. It moves the concept to the slug that `slugify` gives now. It also changes the slug
in each place that points to it: the links of other concepts, attempts, sessions, walks and demos. A slug that a source sets,
such as `sa-term-karaka`, does not change.

In [ ]:
#| export
SLUG_KEYS = ('concept', 'slug', 'focus', 'prereqs', 'related', 'concepts', 'split', 'gaps', 'of')

def _old_slug(s:str) -> str:
    "The slug that abhyasa 0.0.5 made from `s`; empty where it used a random one."
    s = unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode()
    return re.sub(r'[^a-z0-9]+', '-', s.lower()).strip('-')

def _swap(x, moves:dict, key=None):
    "`x` with each slug under a key in `SLUG_KEYS` changed as `moves` says."
    if isinstance(x, dict): return {k: _swap(v, moves, k) for k, v in x.items()}
    if isinstance(x, list): return [_swap(v, moves, key) for v in x]
    return moves.get(x, x) if key in SLUG_KEYS and isinstance(x, str) else x

`reslug` finds the moves first. With `dry_run` it stops there and writes nothing.

In [ ]:
#| export
@patch
def reslug(self:Learning, dry_run:bool=False) -> dict:
    "Move each concept with an old-style slug to the slug from `slugify`, and every slug that points to it. Returns `{old: new}`."
    cs = self.concepts()
    have, moves = set(cs.attrgot('slug')), {}
    for c in cs:
        old = _old_slug(c.name)
        if not (c.slug == old or (not old and re.fullmatch(r'[0-9a-f]{8}', c.slug))) or (new := slugify(c.name)) == c.slug: continue
        base, i = new, 2
        while new in have: new, i = f'{base}-{i}', i + 1
        moves[c.slug] = new; have.add(new)
    if dry_run or not moves: return moves
    for c in cs.filter(lambda c: c.slug in moves):
        m = {**json.loads(first(self.v.t.docs(where='id=?', where_args=[c.doc_id]))['meta'] or '{}'), 'slug': moves[c.slug]}
        self.v.add(concept_text(m), title=c.name, source=_src('concept', moves[c.slug]), kind='concept', meta=m)
        self.v.forget(c.doc_id)
    for r in self.v.t.docs():
        m = json.loads(r['meta'] or '{}')
        if (n := _swap(m, moves)) != m: self.v.set_meta(r['id'], **{k: v for k, v in n.items() if v != m.get(k)})
    return moves

## Groups

A concept can sit in a group under its subject. For example, Magha sits at jyotisha › nakshatras › Magha.
The path is a list of at most two short names in the `group` field. `group_by` tells who set it: `hand` for the learner, `model` for the tutor.
The tutor never moves a concept that the learner moved by hand. An old vault has no groups, so each concept shows under "Not sorted yet".

`clean_group` makes a path safe to keep and to show. Each name has only letters, digits, spaces and hyphens, and at most `GROUP_LEN` characters.

In [ ]:
#| export
GROUP_DEPTH, GROUP_LEN = 2, 40   # the most levels in a path, and the most characters in one name

def _plain(s:str) -> str:
    "`s` with each character that is not a letter, a digit, a mark, a space or a hyphen made a space."
    return ''.join(ch if ch.isalnum() or ch in ' -' or unicodedata.category(ch)[0] == 'M' else ' ' for ch in s)

def clean_group(path, subject:str='') -> list:
    "`path` as at most `GROUP_DEPTH` short plain names. A text splits at `/`, `>` or `›`. Empty parts, repeats and a first part that is the subject drop."
    if isinstance(path, str): path = re.split(r'[/>›]', path)
    if not isinstance(path, (list, tuple)): return []
    out = []
    for p in path:
        s = ' '.join(_plain(p).split())[:GROUP_LEN].strip(' -') if isinstance(p, str) else ''
        if not s or s.lower() in [o.lower() for o in out] or (not out and s.lower() == (subject or '').lower()): continue
        out.append(s)
    return out[:GROUP_DEPTH]

`set_group` moves one concept. A move by hand marks it. A move by the model does not change a concept with the mark. An empty path by hand also keeps the mark: the learner chose no group, so the tutor does not sort the concept again.
`groups` lists the paths in use under one subject, with their parents, so a new concept can go in a group that is there.

In [ ]:
#| export
@patch
def set_group(self:Learning, slug:str, path, by:str='hand') -> AttrDict:
    "Put concept `slug` in the group `path`, set `by` the learner (`hand`) or the tutor (`model`)."
    with self._lock:
        c = self.concept(slug)
        if c is None: raise KeyError(f'no concept {slug!r}')
        if by != 'hand' and c.get('group_by') == 'hand': return c
        g = clean_group(path, c.subject)
        return self.update(slug, group=g, group_by='hand' if by == 'hand' else by if g else None)   # `by`: hand, model, sort (a saved sort) or placed (a new concept)

def _paths(*lists) -> list:
    "The group paths in `lists`, each one time and with its parents, sorted. Case does not count."
    seen = {}
    for g in (g for l in lists for g in l):
        for i in range(1, len(g) + 1): seen.setdefault(tuple(x.lower() for x in g[:i]), list(g[:i]))
    return [seen[k] for k in sorted(seen)]

def _in_use(self:Learning, subject:str='') -> list:
    "The group paths that concepts of `subject` have now."
    return _paths(clean_group(c.get('group'), c.subject) for c in self.concepts(subject or None) if (c.subject or '') == (subject or ''))

def _groups_doc(self:Learning, subject:str='') -> dict|None:
    "The meta of the group list of `subject`: `groups` (the kept paths) and `hand` (the paths that you made or renamed); None when there is no list."
    r = first(self.v.t.docs(where='source=?', where_args=[_src('groups', slugify(subject or 'general'))]))
    return json.loads(r['meta'] or '{}') if r else None

def _write_groups(self:Learning, subject:str, keep:list, hand:list):
    "Write the group list of `subject`: the kept paths and the hand paths."
    src, m = _src('groups', slugify(subject or 'general')), dict(subject=subject or '', groups=keep, hand=hand)
    r = self.v.add('; '.join(' › '.join(g) for g in keep) or 'no groups', title=f"Groups of {subject or 'general'}", source=src, kind='groups', meta=m)
    if not r.get('changed'): self.v.set_meta(r['doc_id'], **m)

@patch
def saved_groups(self:Learning, subject:str='') -> list|None:
    "The group paths kept for `subject` when a sort was saved; None when none was."
    m = _groups_doc(self, subject)
    return m.get('groups', []) if m is not None else None

@patch
def hand_groups(self:Learning, subject:str='') -> list:
    "The group paths of `subject` that you made or renamed. The tutor never renames or drops them."
    return (_groups_doc(self, subject) or {}).get('hand', [])

@patch
def save_groups(self:Learning, subject:str='', groups=None, replace:bool=False) -> list:
    "Keep the group paths of `subject`: `groups` (default: the ones in use) added to the ones kept before. A kept group is never dropped by the tutor. With `replace`, the list becomes `groups` and the hand groups. Return the list."
    with self._lock:
        hand, new = self.hand_groups(subject), _in_use(self, subject) if groups is None else groups
        keep = _paths(new, hand) if replace else _paths(self.saved_groups(subject) or [], new, hand)
        _write_groups(self, subject, keep, hand)
        return keep

@patch
def groups(self:Learning, subject:str='') -> list:
    "The group paths of `subject`: the kept ones and the ones in use, each one time and with its parents, sorted."
    return _paths(self.saved_groups(subject) or [], _in_use(self, subject))

@patch
def unsorted(self:Learning, subject:str=None) -> L:
    "The concepts that the tutor may place: no group, and not kept out of the groups by hand."
    return self.concepts(subject).filter(lambda c: not c.get('group') and c.get('group_by') != 'hand')

@patch
def unplace(self:Learning, slug:str) -> AttrDict:
    "Undo a placement by the tutor: no group, and the tutor does not place the concept by itself again. A group that you set, or that a saved sort set, stays."
    with self._lock:
        c = self.concept(slug)
        if c is None: raise KeyError(f'no concept {slug!r}')
        return self.update(slug, group=[], group_by='undone') if c.get('group_by') == 'placed' else c

@patch
def place(self:Learning, slug:str, path) -> bool:
    "Place a new concept in the group `path`, as the tutor. One step under a lock: it looks at the concept again, and changes nothing when the concept is gone, has a group, was kept out of the groups by hand, or was undone. Return whether it placed the concept."
    with self._lock:
        c = self.concept(slug)
        if c is None or c.get('group') or c.get('group_by') in ('hand', 'undone') or not (g := clean_group(path, c.subject)): return False
        self.update(slug, group=g, group_by='placed')
        return True

### Sort by hand

You can sort the concepts yourself. `rename` gives a concept a new name and marks it `named_by='hand'`, so a new extract never renames it. The old name stays as an alias, so the find box still finds it.
`rename_group` gives a group a new name. Each concept in the group, or in a group below it, gets the new name and `group_by='hand'`. A name that is the name of a group beside it, in any case and without marks, joins the two groups.
`new_group` adds an empty group. `hand_groups` lists the groups that you made or renamed. "Sort everything again" keeps them.
`snapshot` and `restore` are the Undo of one change by hand: `snapshot` keeps the fields that a change by hand can touch, and `restore` writes them back.

In [ ]:
#| export
def fold(s) -> str:
    "`s` in lower case with no marks, so `Maghā` and `MAGHA` are one name."
    return ''.join(ch for ch in unicodedata.normalize('NFKD', str(s or '')) if not unicodedata.combining(ch)).lower()

def _key(g): return tuple(fold(x) for x in g)

def under(g, path) -> bool:
    "Whether the group `g` is the group `path` or a group below it. Case and marks do not count."
    return bool(path) and len(g or []) >= len(path) and _key(g[:len(path)]) == _key(path)

def _uniq(gs) -> list:
    "The paths `gs`, each one time, in their order. Case and marks do not count."
    seen = {}
    for g in gs: seen.setdefault(_key(g), list(g))
    return list(seen.values())

def _spell(gs, g) -> list:
    "`g` with each level spelled as a path in `gs` that is the same name in another case or with marks."
    out = []
    for x in g: out.append(next((p[-1] for p in gs if len(p) == len(out) + 1 and _key(p) == _key([*out, x])), x))
    return out

def _of(self:Learning, subject:str='') -> list:
    "The concepts of `subject` only ('' is the concepts with no subject)."
    return [c for c in self.concepts(subject or None) if (c.subject or '') == (subject or '')]

@patch
def rename(self:Learning, slug:str, name:str) -> AttrDict:
    "Give concept `slug` the name `name`, set by hand: a new extract never renames it. The old name becomes an alias."
    n = ' '.join(str(name or '').split())[:120]
    if not usable_name(n): raise ValueError('Use two letters or digits or more in a name.')
    with self._lock:
        c = self.concept(slug)
        if c is None: raise KeyError(f'no concept {slug!r}')
        if n == c.name: return self.update(slug, named_by='hand')
        return self.put(n, c.statement, c.subject, slug=slug, aliases=[c.name], named_by='hand')   # `put` makes the search text and the title again

@patch
def rename_group(self:Learning, subject:str, path, name:str) -> list:
    "Give the group `path` of `subject` the last name `name`, set by hand. Its concepts and the groups below it move with it. A name of a group beside it joins the two. Return the slugs changed."
    with self._lock:
        path = _spell(self.groups(subject), clean_group(path, subject))
        new = clean_group([*path[:-1], name], subject)
        if not path or len(new) != len(path): raise ValueError('Use letters or digits in a group name.')
        new = _spell([g for g in self.groups(subject) if _key(g) != _key(path)], new)   # a name of a group beside it: the two groups join
        moved = []
        for c in _of(self, subject):
            if under(g := clean_group(c.get('group'), c.subject), path): self.update(c.slug, group=[*new, *g[len(path):]], group_by='hand'); moved.append(c.slug)
        mv = lambda gs: [[*new, *g[len(path):]] if under(g, path) else list(g) for g in gs]
        _write_groups(self, subject, _paths(mv(self.saved_groups(subject) or []), [new]), _uniq([*mv(self.hand_groups(subject)), new]))
        return moved

@patch
def new_group(self:Learning, subject:str, path) -> list:
    "Add the empty group `path` to `subject`, made by hand. Return the group list."
    with self._lock:
        g = _spell(self.groups(subject), clean_group(path, subject))
        if not g: raise ValueError('Use letters or digits in a group name.')
        _write_groups(self, subject, _paths(self.saved_groups(subject) or [], [g]), _uniq([*self.hand_groups(subject), g]))
        return self.groups(subject)

@patch
def unmark_groups(self:Learning, subject:str, paths) -> list:
    "Take the hand mark off the groups `paths` of `subject`, so the tutor can sort them again. Return the hand groups."
    with self._lock:
        m = _groups_doc(self, subject) or {}
        hand = [g for g in m.get('hand', []) if _key(g) not in {_key(p) for p in paths}]
        _write_groups(self, subject, m.get('groups', []), hand)
        return hand

_HELD = ('name', 'named_by', 'aliases', 'group', 'group_by', 'subject', 'subject_by')

@patch
def snapshot(self:Learning, slugs=(), subjects=()) -> AttrDict:
    "The fields that a change by hand can touch, for the concepts `slugs` and the group lists of `subjects`, so `restore` can write them back."
    return AttrDict(cs={s: {k: c.get(k) for k in _HELD} for s in slugs if (c := self.concept(s))},
                    groups={s: _groups_doc(self, s) or {} for s in subjects})

@patch
def restore(self:Learning, snap) -> list:
    "Write back the fields of `snap` (from `snapshot`). A concept that is gone stays gone. Return the slugs written."
    with self._lock:
        for s, m in snap['groups'].items(): _write_groups(self, s, m.get('groups', []), m.get('hand', []))
        out = []
        for s, m in snap['cs'].items():
            if (c := self.concept(s)) is None: continue
            self.update(s, **{**m, 'name': c.name, 'named_by': None})
            if c.name != m['name']: self.put(m['name'], c.statement, slug=s)   # `put` makes the search text and the title again
            self.update(s, named_by=m['named_by']); out.append(s)
        return out

`set_subject` gives a subject to a concept. `readable_name` and `repair_names` make a readable name for a concept whose name is its slug.

In [ ]:
#| export
def clean_subject(s) -> str:
    "`s` as one short plain lower-case name; '' when it has none."
    return ' '.join(_plain(s).split())[:GROUP_LEN].strip(' -').lower() if isinstance(s, str) else ''

@patch
def set_subject(self:Learning, slug:str, subject:str, by:str='model') -> AttrDict:
    "Give concept `slug` a subject, set `by` the learner (`hand`) or the tutor (`model`). The tutor sets only an empty subject, and never on a concept that you moved by hand."
    c = self.concept(slug)
    if c is None: raise KeyError(f'no concept {slug!r}')
    s = clean_subject(subject)
    if by != 'hand' and (c.get('group_by') == 'hand' or c.subject or not s): return c
    return self.put(c.name, c.statement, s, slug=slug, subject_by=by)   # `put` makes the search text and the title again

def readable_name(name, statement:str='') -> str:
    "`name` as words: a name that is a slug with hyphens, or an empty name (words from `statement`), becomes a short sentence-case name. Else `name`."
    n = ' '.join(str(name or '').split())
    if n and not (' ' not in n and '-' in n and n == slugify(n)): return n
    n = ' '.join((n or ' '.join(str(statement or '').split()[:6])).replace('-', ' ').split()).rstrip('.,;:')
    return n[:1].upper() + n[1:]

@patch
def repair_names(self:Learning) -> list:
    "One time for each shelf, give each concept whose name is its own slug, with hyphens, a readable name. Return the slugs that changed. A name that differs from the slug, or that you set by hand (`named_by`), stays."
    if first(self.v.t.docs(where='source=?', where_args=[_src('marker', 'names')])): return []
    out = [c.slug for c in self.concepts() if c.name == c.slug and '-' in c.name and c.get('named_by') != 'hand']   # `named_by='hand'`: no rename path yet, the editors of 0.0.13 set it
    for s in out: c = self.concept(s); self.put(readable_name(s), c.statement, c.subject, slug=s)   # `put` makes the search text and the title again
    self.v.add('The names of the concepts were repaired.', title='Names repaired', source=_src('marker', 'names'), kind='marker')
    return out

`tree` puts concepts in a tree for the Concepts page: a node for each subject, then its groups, then their groups. A node has its `name`, its `path` under the subject, `n` (the concepts in it and below it), `kids` (its groups by name) and `cs` (its own concepts). A concept with no group is in `cs` of its subject. Two names that differ only in case are one group.

In [ ]:
#| export
def _node(name, path): return AttrDict(name=name, path=path, kids={}, cs=[])

def tree(concepts, groups=None) -> list:
    "The subject nodes of `concepts`, sorted by name, each with its groups and the count of concepts below it. `groups` (`{subject: paths}`) adds groups with no concept, under a subject that shows."
    root = {}
    def walk(n, g):
        for x in g: n = n.kids.setdefault(x.lower(), _node(x, [*n.path, x]))
        return n
    for c in concepts:
        s = c.get('subject') or ''
        walk(root.setdefault(s.lower(), _node(s, [])), clean_group(c.get('group'), s)).cs.append(c)
    for s, gs in (groups or {}).items():
        if (s or '').lower() in root:
            for g in gs: walk(root[(s or '').lower()], g)
    def done(n):
        n.kids = [done(k) for _, k in sorted(n.kids.items())]
        n.cs = sorted(n.cs, key=lambda c: (c.get('order', 1e9), c['name'].lower()))
        n.n = len(n.cs) + sum(k.n for k in n.kids)
        return n
    return [done(n) for _, n in sorted(root.items())]

## Attempts

Each answer is one attempt document. Thus search can find what the learner said. The tutor can also find old questions and not ask them again.

In [ ]:
#| export
def attempt_text(a) -> str:
    "The searchable text of an attempt: the question, the answer, the think-aloud and the feedback."
    parts = [f"Question: {a['question']}", a.get('answer') and f"Answer: {a['answer']}",
             a.get('thinking') and f"Thinking: {a['thinking']}", a.get('feedback') and f"Feedback: {a['feedback']}"]
    return '\n\n'.join(p for p in parts if p)

`record` saves one attempt. The vault joins two sources with the same text into one document. So `record` asks for a new document each time (`force=True`): two attempts with the same words stay two records. Each one has its own source, `learning://attempt/<id>`.

In [ ]:
#| export
@patch
def record(self:Learning, concept:str, question:str, answer:str='', thinking:str='', correct:bool=False,
           confidence:int=None, depth:int=1, kind:str='explain', hints:int=0, dont_know:bool=False, grade:int=None,
           feedback:str='', first_error:str='', misconception:str='', session:str='', **meta) -> AttrDict:
    "File one attempt as a vault document, so the learner can search what they said."
    a = AttrDict(id=uuid.uuid4().hex[:12], concept=concept, question=question, answer=answer, thinking=thinking,
                 correct=correct, confidence=confidence, depth=depth, kind=kind, hints=hints, dont_know=dont_know,
                 grade=grade, feedback=feedback, first_error=first_error, misconception=misconception,
                 session=session, at=time.time(), **meta)
    c = self.concept(concept)
    self.v.add(attempt_text(a), title=f"{c.name if c else concept}: {question[:80]}", source=_src('attempt', a.id),
               kind='attempt', meta=dict(a), force=True)   # its own document, also when an older attempt has the same text
    return a

`attempts` reads them back, with the newest first.

In [ ]:
#| export
@patch
def attempts(self:Learning, concept:str=None, session:str=None, limit:int=50, since:float=None) -> L:
    "Recent attempts, newest first, for one concept or one session; with `since`, only the attempts from that time on."
    w, args = [], []
    if since is not None: w.append("json_extract(meta, '$.at')>=?"); args.append(since)
    if concept: w.append("json_extract(meta, '$.concept')=?"); args.append(concept)
    if session: w.append("json_extract(meta, '$.session')=?"); args.append(session)
    rs = L(self._docs('attempt', ' and '.join(w), args)).map(lambda r: AttrDict(json.loads(r['meta'])))
    return L(sorted(rs, key=lambda a: -a.at)[:limit]).map(lambda a: AttrDict({k: v for k, v in a.items() if k not in _SKIP}))

`mark_attempt` adds fields to a saved attempt, for example a lesson that you asked for later. `add_gap` keeps a gap that you named on a concept.

In [ ]:
#| export
@patch
def mark_attempt(self:Learning, aid:str, **kv) -> dict:
    "Merge `kv` into the meta of attempt `aid`."
    r = first(self.v.t.docs(where='source=?', where_args=[_src('attempt', aid)]))
    if r is None: raise KeyError(f'no attempt {aid!r}')
    return self.v.set_meta(r['id'], **kv)

@patch
def add_gap(self:Learning, slug:str, gap:str):
    "Record that you said you do not know concept `gap` while you learned `slug`."
    c = self.concept(slug)
    if c is None: raise KeyError(f'no concept {slug!r}')
    if gap != slug and gap not in (c.get('gaps') or []): self.update(slug, gaps=[*(c.get('gaps') or []), gap])

## Sessions

A session summary is one document. A new summary with the same id replaces the old one.

In [ ]:
#| export
@patch
def log_session(self:Learning, sid:str, summary:str, **meta):
    "File a session, and replace an earlier one with the same id. The meta changes also when the summary text does not."
    m = dict(id=sid, summary=summary, **meta)
    r = self.v.add(summary, title=f'Session {sid}', source=_src('session', sid), kind='session', meta=m)
    if not r.get('changed'): self.v.set_meta(r['doc_id'], **m)
    return r

@patch
def session(self:Learning, sid:str) -> AttrDict|None:
    "One filed session by id, as its meta."
    r = first(self.v.t.docs(where='source=?', where_args=[_src('session', sid)]))
    return AttrDict({**json.loads(r['meta'] or '{}'), 'doc_id': r['id']}) if r else None

@patch
def sessions(self:Learning, limit:int=100, journey:str=None) -> L:
    "The filed sessions, the most recently changed first; with `journey`, only the sessions of that walk."
    w, args = ("json_extract(meta, '$.journey')=?", [journey]) if journey else ('', [])
    rs = L(self._docs('session', w, args)).map(lambda r: AttrDict({**json.loads(r['meta'] or '{}'), 'doc_id': r['id']}))
    return L(sorted(rs, key=lambda s: -(s.get('updated') or s.get('end') or s.get('start') or 0))[:limit])

## Search and graph

`search` uses the hybrid search of the vault. Each hit has the kind, title and meta of its document. The method stats, the marker of a repair, a session with no answers and an open question stay out of it. Two attempts with the same text show as one hit. `graph` gives the nodes and links for the graph view.

In [ ]:
#| export
def _answers(m) -> int:
    "How many answers a filed session has: its log, or `asked` for an old one with no log."
    return len(m['log']) if 'log' in m else m.get('asked', 0)

@patch
def search(self:Learning, q:str, kind:str=None, limit:int=10) -> L:
    """Hybrid search over the shelf. Each hit also has the `kind`, `title` and `meta` of its document, so a page can show it
    by kind. The method stats, the marker of a repair, a session with no answers and an open question stay out: they have no text that you wrote or learned,
    and an open question has its answer key."""
    n = limit + 10
    while True:   # fetch more until there are `limit` real hits, or the search has no more
        hs = L(self.v.search(q, limit=n, kind=kind))
        ids = list(dict.fromkeys(h['doc_id'] for h in hs))
        rows = {r['id']: r for r in self.v.t.docs(where=f"id in ({','.join('?' * len(ids))})", where_args=ids)} if ids else {}
        out, seen = L(), set()
        for h in hs:
            if (r := rows.get(h['doc_id'])) is None: continue
            m = json.loads(r['meta'] or '{}')
            if r['kind'] in ('stats', 'marker', 'groups', 'open-q') or (r['kind'] == 'session' and not _answers(m)): continue
            if r['kind'] == 'attempt':   # two attempts with the same words are two records, but one hit
                if (k := (r['title'], h.get('snippet'))) in seen: continue
                seen.add(k)
            out.append({**h, 'kind': r['kind'], 'title': r['title'], 'meta': m})
        if len(out) >= limit or len(hs) < n: return out[:limit]
        n *= 4

@patch
def graph(self:Learning, subject:str=None) -> dict:
    "Nodes and links (prereq, related, gap) for a graph view. A link to a slug that has no concept is dropped."
    cs = self.concepts(subject)
    have = set(cs.attrgot('slug'))
    nodes = [dict(id=c.slug, name=c.name, subject=c.subject, floor=c.floor, ceiling=c.ceiling, due=c.due, seen=c.seen, group=clean_group(c.get('group'), c.subject)) for c in cs]
    links = [dict(source=p, target=c.slug, kind='prereq') for c in cs for p in c.prereqs if p in have]
    links += [dict(source=c.slug, target=r, kind='related') for c in cs for r in c.related if r in have and c.slug < r]
    links += [dict(source=g, target=c.slug, kind='gap') for c in cs for g in c.get('gaps') or () if g in have]
    return dict(nodes=nodes, links=links)

## A shelf for tests

Each test uses a vault in a temporary folder with the offline hashing encoder.
The tests never open the real vault of the learner.

In [ ]:
#| export
def temp_learning(**kw) -> Learning:
    "A `Learning` on a new vault in a temporary folder, with the offline hashing encoder."
    import tempfile
    return Learning(Vault(str(Path(tempfile.mkdtemp())/'vault.db'), offline=True), **kw)

## Tests

`put` makes a concept with the start state and the slug from its name.

In [ ]:
lr = temp_learning()
c = lr.put('Reciprocal rank fusion', 'Merge ranked lists by summing 1/(k + rank) for each item.', 'ml', prereqs=['ranking'], aliases=['RRF'])
test_eq(c.slug, 'reciprocal-rank-fusion')
test_eq(c.floor, 0)
test_eq(c.prereqs, ['ranking'])

`update` changes the state.

In [ ]:
c = lr.update(c.slug, floor=3, stability=2.)
test_eq((c.floor, c.stability), (3, 2.))

A second `put` keeps the learner state and merges the lists, also when the text changes.

In [ ]:
c = lr.put('Reciprocal rank fusion', 'Merge ranked lists: score is the sum of 1/(k + rank).', related=['bm25'], aliases=['RRF', 'rank fusion'])
test_eq(c.floor, 3)
test_eq(c.subject, 'ml')
test_eq(c.aliases, ['RRF', 'rank fusion'])
test_eq(c.related, ['bm25'])
test_eq(len(lr.concepts()), 1)

`concepts` and `subjects` filter by subject. `graph` keeps only the links between known concepts.

In [ ]:
lr.put('Ranking', 'Order items by a score.', 'ml')
lr.put('Sandhi', 'Sounds change where words meet.', 'sanskrit')
test_eq(len(lr.concepts('ml')), 2)
test_eq({r['subject'] for r in lr.subjects()}, {'ml', 'sanskrit'})
g = lr.graph()
test_eq(g['links'], [dict(source='ranking', target='reciprocal-rank-fusion', kind='prereq')])

`record` saves an attempt. `attempts` and `search` find it again.

In [ ]:
a = lr.record('reciprocal-rank-fusion', 'Why does RRF ignore raw scores?', answer='Because scores from different systems are not comparable.',
              thinking='bm25 and cosine have different scales', correct=True, confidence=80, depth=3)
test_eq(lr.attempts('reciprocal-rank-fusion')[0].id, a.id)
test_eq(lr.attempts('ranking'), [])
test(lr.search('scales comparable', kind='attempt'), [], lambda a,b: len(a) > 0)

`drop` removes a concept.

In [ ]:
lr.drop('ranking')
test_is(lr.concept('ranking'), None)

The search fetches more hits until it has `limit` real hits, or the shelf has no more. Here 30 sessions with no answers match the words best. Three concepts match only one word. All three concepts come back.

In [ ]:
sl = temp_learning()
for k in range(30): sl.log_session(f'empty{k}', f'zebra quagga okapi zebra quagga okapi {k}', start=1., log=[])
for k in range(3): sl.put(f'Giraffe {k}', f'A tall animal {k}, once next to an okapi.', 'zoo')
test_eq(len(sl.search('zebra quagga okapi', limit=3)), 3)

In [ ]:
lr.log_session('s1', '1 of 2 passed.', start=1., updated=2., asked=2)
lr.log_session('s1', '1 of 2 passed.', start=1., updated=3., asked=2, ended=3.)   # the same text: the meta still changes
test_eq((lr.session('s1').updated, lr.session('s1').ended, lr.session('s1').summary), (3., 3., '1 of 2 passed.'))
test_eq(len(lr.sessions()), 1)

`sessions` lists the most recently changed session first.

In [ ]:
lr.log_session('s2', 'Started.', start=5., updated=5.)
test_eq(lr.sessions().attrgot('id'), ['s2', 's1'])
test_is(lr.session('nope'), None)

With `journey`, `sessions` reads only the sessions of that walk, in the store.

In [ ]:
lr.log_session('s3', 'A walk.', start=6., journey='j1')
test_eq(lr.sessions(journey='j1').attrgot('id'), ['s3'])
test_eq(len(lr.sessions()), 3)

`search` leaves out a session with no answers and the method stats. The session is still filed, so you can continue it. Each hit names its kind.

In [ ]:
lr.log_session('s4', '0 of 0 passed. Concepts: none.', start=7., log=[])
lr.v.add('How you learn: method counts, passed.', title='How you learn', source='stats:test', kind='stats')
hs = lr.search('passed', limit=20)
test_eq([h for h in hs if h['kind'] == 'stats' or h['meta'].get('id') == 's4'], [])
test_eq('s1' in [h['meta'].get('id') for h in hs if h['kind'] == 'session'], True)   # a session with answers stays
test_eq(lr.session('s4').id, 's4')

The search fetches more hits until it has `limit` real hits, or the shelf has no more. Here 30 sessions with no answers match the words first, and three concepts match them too. All three concepts come back.

`concept_by_doc` gives the concept of a search hit. It gives None for a hit that is not a concept.

In [ ]:
cl = temp_learning()
cl.put('Reciprocal rank fusion', 'Merge ranked lists by summing 1/(k + rank).', 'ml')
cl.record('reciprocal-rank-fusion', 'Why ranks?', answer='scales differ')
test_eq(cl.concept_by_doc(cl.search('rank fusion', kind='concept')[0]['doc_id']).slug, 'reciprocal-rank-fusion')
test_is(cl.concept_by_doc(cl.search('scales differ', kind='attempt')[0]['doc_id']), None)

`slugify` reads Devanagari and Telugu, keeps C, C++ and C# apart, and gives the same hash slug for the same unreadable name.

In [ ]:
test_eq([slugify(s) for s in ('saṃdhi', 'संधि', 'గుణ', 'C', 'C++', 'C#', 'R&D')], ['samdhi', 'samdhi', 'guna', 'c', 'c-plus-plus', 'c-sharp', 'r-and-d'])
test_eq(slugify('学习'), slugify('学习'))
assert slugify('学习').startswith('c-')

This test makes a vault for `reslug` as abhyasa 0.0.5 left it: C# saved as `c`, a Devanagari name with a random slug, and links, an attempt and a session that use these slugs.

In [ ]:
ol = temp_learning()
ol.put('C#', 'A language from Microsoft.', 'code', slug='c')
ol.put('संधि', 'Sounds change where words meet.', 'sanskrit', slug='a1b2c3d4')
ol.put('karaka term', 'A set slug.', 'sanskrit', slug='sa-term-karaka')
ol.put('Ranking', 'Order items by a score.', 'ml', prereqs=['c'], related=['a1b2c3d4', 'sa-term-karaka'])
ol.record('c', 'What is C#?', answer='c')
ol.log_session('s1', 'One question.', concepts=['c'], log=[dict(concept='c', question='What is C#?')], focus='a1b2c3d4')

A dry run lists the moves and changes nothing. A slug that a source sets is not in the list.

In [ ]:
test_eq(ol.reslug(dry_run=True), {'c': 'c-sharp', 'a1b2c3d4': 'samdhi'})
assert ol.concept('c') is not None and ol.concept('c-sharp') is None

The real run moves the concepts and every slug that points to them. An answer that is the text `c` stays as it is. A second run has nothing to do.

In [ ]:
test_eq(ol.reslug(), {'c': 'c-sharp', 'a1b2c3d4': 'samdhi'})
test_eq((ol.concept('c'), ol.concept('a1b2c3d4')), (None, None))
test_eq((ol.concept('c-sharp').name, ol.concept('samdhi').statement), ('C#', 'Sounds change where words meet.'))
r = ol.concept('ranking')
test_eq((r.prereqs, r.related), (['c-sharp'], ['samdhi', 'sa-term-karaka']))
a = ol.attempts('c-sharp')[0]
test_eq((a.concept, a.answer), ('c-sharp', 'c'))
s = ol.session('s1')
test_eq((s.concepts, s.log[0]['concept'], s.focus), (['c-sharp'], 'c-sharp', 'samdhi'))
test_eq(ol.reslug(), {})

`mark_attempt` changes a saved attempt. `add_gap` keeps each gap one time, and `graph` shows it as a `gap` link.

In [ ]:
gl = temp_learning()
gl.put('Functions and the neuron', 'z = w . x + b.', 'ml'); gl.put('Weighted sum', 'Add each input times its weight.', 'ml')
a = gl.record('functions-and-the-neuron', 'Why does z double?', answer='2z')
test_eq(gl.mark_attempt(a.id, taught=True)['taught'], True)
test_eq(gl.attempts('functions-and-the-neuron')[0].taught, True)
gl.add_gap('functions-and-the-neuron', 'weighted-sum'); gl.add_gap('functions-and-the-neuron', 'weighted-sum')
test_eq(gl.concept('functions-and-the-neuron').gaps, ['weighted-sum'])
assert dict(source='weighted-sum', target='functions-and-the-neuron', kind='gap') in gl.graph()['links']

Two attempts with the same text are two records. The vault does not join them, so each one counts in your history and in the misses that the tutor counts. A field added later goes on the right attempt.

In [ ]:
ld = temp_learning(); ld.put('Ranking', 'Order items by a score.', 'ml')
a1, a2 = [ld.record('ranking', 'What is a rank?', 'a score', feedback='Look at positions.', final=True) for _ in range(2)]
test_eq(ld.attempts('ranking').attrgot('id'), [a2.id, a1.id])
ld.mark_attempt(a2.id, taught=1)
test_eq([a.get('taught') for a in ld.attempts('ranking')], [1, None])

A search shows the same attempt text only once.

In [ ]:
test_eq(len([h for h in ld.search('What is a rank', kind='attempt') if h['kind'] == 'attempt']), 1)

## Groups

An old vault has no groups. A concept loads with an empty path, and no one set it.

In [ ]:
gv = temp_learning()
m = gv.put('Magha', 'The tenth nakshatra.', 'jyotisha')
test_eq((m.get('group'), m.get('group_by')), (None, None))

`clean_group` keeps at most two short plain names. It drops marks, empty parts, a repeat and a first part that is the subject. A text splits at `/`, `>` or `›`.

In [ ]:
test_eq(clean_group(['Nakshatras', 'Padas', 'More']), ['Nakshatras', 'Padas'])
test_eq(clean_group(' nakshatras  /  padas! '), ['nakshatras', 'padas'])
test_eq(clean_group(['Nakshatras!', '  deva   gana? ']), ['Nakshatras', 'deva gana'])
test_eq(clean_group(['jyotisha', 'nakshatras'], 'jyotisha'), ['nakshatras'])
test_eq(clean_group(['', 'Nakshatras', 'nakshatras', 3]), ['Nakshatras'])
test_eq(clean_group('saṃdhi › स्वर'), ['saṃdhi', 'स्वर'])
test_eq(len(clean_group(['x' * 90])[0]), GROUP_LEN)
test_eq((clean_group(None), clean_group({'a': 1}), clean_group('')), ([], [], []))

`set_group` by the model moves a concept. A move by hand sets `group_by='hand'`. After that, a move by the model changes nothing.

In [ ]:
test_eq(gv.set_group('magha', ['nakshatras'], by='model').group, ['nakshatras'])
m = gv.set_group('magha', 'Nakshatras / Gana')
test_eq((m.group, m.group_by), (['Nakshatras', 'Gana'], 'hand'))
test_eq(gv.set_group('magha', ['rasis'], by='model').group, ['Nakshatras', 'Gana'])

A new `put` with a group, as from a new source, does not move a concept that you moved by hand. It keeps the learner state.

In [ ]:
m = gv.put('Magha', 'The tenth nakshatra, ruled by Ketu.', 'jyotisha', group=['rasis'], group_by='model')
test_eq((m.group, m.group_by, m.statement), (['Nakshatras', 'Gana'], 'hand', 'The tenth nakshatra, ruled by Ketu.'))

An empty path by hand takes the concept out of its group and keeps the mark. So the tutor does not sort it again, and a move by the model changes nothing.

In [ ]:
m = gv.set_group('magha', '')
test_eq((m.group, m.group_by), ([], 'hand'))
test_eq(gv.set_group('magha', ['rasis'], by='model').group, [])

`groups` lists the paths in use under one subject, with their parents. A path in another subject does not show.

`set_subject` gives a subject to a concept that has none. A move by the tutor never changes a concept that you moved by hand, and it never replaces a subject. A move by hand always works.

In [ ]:
sj = temp_learning()
sj.put('Alpha', 'One.'); sj.put('Beta', 'Two.'); sj.put('Gamma', 'Three.', 'ml')
sj.set_group('beta', 'mine')
test_eq(sj.set_subject('alpha', ' Star  Maps! ').subject, 'star maps')
test_eq(sj.concept('alpha').subject_by, 'model')
test_eq(sj.set_subject('beta', 'stars').subject, '')                 # moved by hand: the tutor does not touch it
test_eq(sj.set_subject('gamma', 'stars').subject, 'ml')              # it has a subject
test_eq(sj.set_subject('gamma', 'stars', by='hand').subject, 'stars')

A name that is its own slug, with hyphens, gets a readable name one time. A name that differs from the slug stays. The empty name of the finder gets words from the statement.

In [ ]:
test_eq(readable_name('attention-heads-and-anaphora-resolution'), 'Attention heads and anaphora resolution')
test_eq(readable_name('Softmax'), 'Softmax'); test_eq(readable_name('softmax'), 'softmax'); test_eq(readable_name('my-Note'), 'my-Note')
test_eq(readable_name('', 'A rank is a position in a list. More.'), 'A rank is a position in')
rn = temp_learning()
rn.put('attention-heads-and-anaphora-resolution', 'Heads.', 'ml', slug='attention-heads-and-anaphora-resolution')
rn.put('my-note', 'Mine.', 'ml', slug='note')
rn.put('foo-bar', 'Typed.', 'ml', slug='foo-bar', named_by='hand')
def old_vault(l):
    "Forget the mark of the repair, as in a vault from before it."
    l.v.forget(first(l.v.t.docs(where='source=?', where_args=['learning://marker/names']))['id'])
old_vault(rn)
test_eq(Learning(rn.vault).concept('attention-heads-and-anaphora-resolution').name, 'Attention heads and anaphora resolution')
test_eq((rn.concept('note').name, rn.concept('foo-bar').name), ('my-note', 'foo-bar'))   # a name that differs from the slug, or that you typed, stays
test_eq(rn.repair_names(), [])

The repair runs one time: a mark on the shelf says that it ran. A vault that cannot be read or changed does not break the load. A change of name or subject also makes the search text and the title again.

In [ ]:
from unittest.mock import patch as mock_patch

In [ ]:
rn.put('late-slug-name', 'Late.', 'ml', slug='late-slug-name')
test_eq(rn.repair_names(), [])                                        # the mark is set: it does not look again
test_eq(rn.concept('late-slug-name').name, 'late-slug-name')
old_vault(rn)
with mock_patch.object(Learning, 'repair_names', side_effect=OSError('read only')): Learning(rn.vault)   # the load goes on
test_eq(rn.repair_names(), ['late-slug-name'])
doc = lambda l, s: first(l.v.t.docs(where='source=?', where_args=[f'learning://concept/{s}']))
test_eq(doc(rn, 'late-slug-name')['title'], 'Late slug name')
sj.set_subject('gamma', 'cosmos', by='hand')
test_eq(sj.concept('gamma').subject, 'cosmos')
assert sj.concept('gamma').doc_id in [h['doc_id'] for h in sj.v.search('cosmos', 3)]   # the search text has the new subject

The mark of the repair is a document, but search never gives it.

In [ ]:
mt = temp_learning(); mt.put('Names', 'Names of concepts repaired.', 'ml')
assert first(mt.v.t.docs(where='kind=?', where_args=['marker']))
test_eq([h['kind'] for h in mt.search('names concepts repaired')], ['concept'])
test_eq(mt.search('Names repaired', kind='marker'), [])

### Saved groups stay

When you save a sort, the tutor keeps the group list of each subject and does not move a sorted concept again. `save_groups` keeps the list. `groups` gives the saved groups and the groups in use. A saved group stays when no concept is in it. `unsorted` lists the concepts that the tutor may still place: no group, and not set by hand.

In [ ]:
cl = temp_learning()
for n in ('Ashvini', 'Bharani', 'Krittika'): cl.put(n, f'{n}.', 'jyotisha')
test_is(cl.saved_groups('jyotisha'), None)
cl.set_group('ashvini', ['Stars'], by='sort')
test_eq(cl.save_groups('jyotisha'), [['Stars']])
cl.set_group('ashvini', [], by='hand'); cl.set_group('bharani', ['Signs'], by='sort')
test_eq(cl.save_groups('jyotisha'), [['Signs'], ['Stars']])           # a kept group stays when no concept is in it
test_eq(cl.groups('jyotisha'), [['Signs'], ['Stars']])
test_eq(cl.saved_groups('JYOTISHA'), [['Signs'], ['Stars']])
test_eq(cl.concept('bharani').group_by, 'sort')

A concept with a group is sorted. A concept with no group is not sorted, except when you set it by hand to no group. `unplace` is the Undo of a placement by the tutor: the concept has no group, and the tutor does not place it by itself again.

In [ ]:
test_eq(cl.unsorted('jyotisha').attrgot('slug'), ['krittika'])         # ashvini: kept out by hand; bharani: sorted
cl.set_group('krittika', ['Stars'], by='placed')
test_eq(cl.unplace('krittika').group_by, 'undone')
test_eq((cl.concept('krittika').group, cl.unsorted('jyotisha').attrgot('slug')), ([], ['krittika']))
test_eq(cl.unplace('bharani').group, ['Signs'])                         # only a placement by the tutor can be undone
test_eq(cl.unplace('ashvini').group_by, 'hand')

A name with fewer than two characters, or with no letter or digit, is not an idea: `usable_name` says so, and the places that make concepts from a text, a term or a tool call use it. The marker and the group list never show in search.

In [ ]:
assert usable_name('Ab') and usable_name('x2') and usable_name('C#') and not usable_name('x') and not usable_name(' - ') and not usable_name(None)
test_eq(cl.search('Groups of jyotisha', kind='groups'), [])
assert not [h for h in cl.search('Groups of jyotisha') if h['kind'] == 'groups']

In [ ]:
gv.put('Ashvini', 'The first nakshatra.', 'jyotisha', group=['nakshatras', 'deva gana'])
gv.put('Mesha', 'The first rasi.', 'jyotisha', group=['Rasis'])
gv.put('Sandhi', 'Sounds change where words meet.', 'sanskrit', group=['phonology'])
test_eq(gv.groups('jyotisha'), [['nakshatras'], ['nakshatras', 'deva gana'], ['Rasis']])
test_eq(gv.groups('ml'), [])

`tree` puts the concepts under their subject, then their groups. Each node has the count of the concepts in it and below it. A concept with no group is an item of its subject. A group name in another case is the same group.

In [ ]:
gv.put('Bharani', 'The second nakshatra.', 'jyotisha', group=['Nakshatras'])
t = tree(gv.concepts())
test_eq([(s.name, s.n) for s in t], [('jyotisha', 4), ('sanskrit', 1)])
j = t[0]
test_eq([(k.name, k.n) for k in j.kids], [('nakshatras', 2), ('Rasis', 1)])
test_eq([c.name for c in j.cs], ['Magha'])
test_eq([(k.path, [c.name for c in k.cs]) for k in j.kids[0].kids], [(['nakshatras', 'deva gana'], ['Ashvini'])])
test_eq([c.name for c in j.kids[0].cs], ['Bharani'])

### Sort by hand

A renamed group takes its concepts with it. They and the group get the hand mark, so the tutor never moves them.

In [ ]:
hs = temp_learning()
for n in ('Ashvini', 'Bharani'): hs.put(n, f'{n}.', 'jyotisha', group=['nakshatras'])
hs.save_groups('jyotisha')
test_eq(sorted(hs.rename_group('jyotisha', ['Nakshatras'], 'Lunar mansions')), ['ashvini', 'bharani'])
test_eq({(tuple(c.group), c.group_by) for c in hs.concepts('jyotisha')}, {(('Lunar mansions',), 'hand')})
test_eq((hs.hand_groups('jyotisha'), hs.groups('jyotisha')), ([['Lunar mansions']], [['Lunar mansions']]))
assert not hs.unsorted('jyotisha') and not hs.place('ashvini', ['Other'])

A name that is the name of a group beside it, in another case, joins the two groups. The group below a renamed group moves with it.

In [ ]:
hs.put('Pada', 'A quarter.', 'jyotisha', group=['Stars', 'padas']); hs.put('Pada two', 'A quarter.', 'jyotisha', group=['Stars x'])
hs.rename_group('jyotisha', ['Stars x'], 'STARS')
test_eq(hs.concept('pada-two').group, ['Stars'])
hs.rename_group('jyotisha', ['Stars'], 'Asterisms')
test_eq((hs.concept('pada').group, hs.concept('pada-two').group), (['Asterisms', 'padas'], ['Asterisms']))
assert ['Stars'] not in hs.groups('jyotisha') and ['Stars x'] not in hs.groups('jyotisha')
test_fail(lambda: hs.rename_group('jyotisha', ['Asterisms'], '!!!'), contains='letters or digits')

`new_group` adds an empty group that stays. `save_groups(replace=True)` makes the list again and keeps the hand groups. `unmark_groups` takes the mark off.

In [ ]:
hs.new_group('jyotisha', 'Rasis')
assert ['Rasis'] in hs.groups('jyotisha') and ['Rasis'] in hs.hand_groups('jyotisha')
test_eq(hs.save_groups('jyotisha', [['Grahas']], replace=True), [['Asterisms'], ['Grahas'], ['Lunar mansions'], ['Rasis']])
assert ['Asterisms', 'padas'] in hs.groups('jyotisha')   # a group in use still shows
hs.unmark_groups('jyotisha', [['rasis']])
assert ['Rasis'] not in hs.hand_groups('jyotisha') and ['Rasis'] in hs.groups('jyotisha')

A name that you set stays when a new extract puts the concept again. The old name becomes an alias.

In [ ]:
hs.put('Self-attention', 'x', 'ml'); c = hs.rename('self-attention', 'Attention over one sequence')
test_eq((c.name, c.named_by, c.aliases), ('Attention over one sequence', 'hand', ['Self-attention']))
test_eq(hs.put('Self-attention', 'y', 'ml').name, 'Attention over one sequence')
test_fail(lambda: hs.rename('self-attention', ' x '), contains='two letters')

`snapshot` and `restore` undo one change by hand: the groups of the concepts, their names and the group list.

In [ ]:
snap = hs.snapshot(['pada', 'self-attention'], ['jyotisha'])
hs.rename_group('jyotisha', ['Asterisms'], 'Stars again'); hs.rename('self-attention', 'Self focus'); hs.new_group('jyotisha', 'Extra')
test_eq(sorted(hs.restore(snap)), ['pada', 'self-attention'])
test_eq((hs.concept('pada').group, hs.concept('self-attention').name), (['Asterisms', 'padas'], 'Attention over one sequence'))
assert ['Extra'] not in hs.groups('jyotisha') and ['Asterisms'] in hs.saved_groups('jyotisha')

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()